In [2]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_dublin_core.xml", "data/metadata/climate_dataset_dublin_core.xml"),
    ("data/metadata/climate_dataset_datacite.xml", "data/metadata/climate_dataset_datacite.xml"),
    ("data/metadata/climate_dataset_schemaorg.jsonld", "data/metadata/climate_dataset_schemaorg.jsonld"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_dublin_core.xml
Downloaded: data/metadata/climate_dataset_datacite.xml
Downloaded: data/metadata/climate_dataset_schemaorg.jsonld
Bootstrap complete.


# Day 2: Mini Metadata Validation (Teaching Demo)

This notebook implements a **lightweight** metadata completeness check.

**Important:** This is a teaching demo, not a formal validator.

## What we check
- `title` exists
- `creator` exists
- `identifier` exists
- `description` exists
- at least one of `publisher`, `publicationYear`, or `date` exists


In [7]:
from pathlib import Path
import sys
import pandas as pd
from urllib.request import urlretrieve

# --- Start of added code to fix ModuleNotFoundError for 'src' ---
GITHUB_RAW_BASE_FOR_SRC = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_src_files = [
    ("src/__init__.py", "src/__init__.py"),
    ("src/parse_dublin_core.py", "src/parse_dublin_core.py"),
    ("src/parse_datacite.py", "src/parse_datacite.py"),
    ("src/parse_schemaorg.py", "src/parse_schemaorg.py"),
    ("src/validate_metadata.py", "src/validate_metadata.py"),
    ("src/utils.py", "src/utils.py"), # Added src/utils.py to the download list
]

for local_rel, github_rel in required_src_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE_FOR_SRC}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded src file: {local_path}")
# --- End of added code ---


repo_root = Path().resolve()
sys.path.insert(0, str(repo_root))

from src.parse_dublin_core import parse_dublin_core_xml
from src.parse_datacite import parse_datacite_xml
from src.parse_schemaorg import parse_schemaorg_jsonld
from src.validate_metadata import completeness_score

dc_path = "data/metadata/climate_dataset_dublin_core.xml"
dc_meta = parse_dublin_core_xml(dc_path)

dcite_path =  "data/metadata/climate_dataset_datacite.xml"
datacite_meta = parse_datacite_xml(dcite_path)

schema_path = "data/metadata/climate_dataset_schemaorg.jsonld"
schema_meta = parse_schemaorg_jsonld(schema_path)

scores = [completeness_score(dc_meta), completeness_score(datacite_meta), completeness_score(schema_meta)]
scores

[{'standard': 'Dublin Core',
  'score_percent': 100.0,
  'checks': [('title', True),
   ('creator', True),
   ('identifier', True),
   ('description', True),
   ('publisher_or_year', True)]},
 {'standard': 'DataCite',
  'score_percent': 100.0,
  'checks': [('title', True),
   ('creator', True),
   ('identifier', True),
   ('description', True),
   ('publisher_or_year', True)]},
 {'standard': 'schema.org',
  'score_percent': 100.0,
  'checks': [('title', True),
   ('creator', True),
   ('identifier', True),
   ('description', True),
   ('publisher_or_year', True)]}]

In [8]:
score_rows = [
    {"standard": s["standard"], "completeness_score_percent": s["score_percent"]}
    for s in scores
]

score_df = pd.DataFrame(score_rows).sort_values("completeness_score_percent", ascending=False)
score_df

,standard,completeness_score_percent
0,Dublin Core,100.0
1,DataCite,100.0
2,schema.org,100.0


## Interpretation (teaching perspective)

In practice, completeness checks help identify missing fields early.

To improve Reusability, students typically focus on:
- richer `description` and clear variable meaning,
- clear `publisher` and consistent `identifier` usage,
- and any documentation that helps others interpret the dataset correctly.
